# 05 · Mechanism: where does the hidden knowledge live?
### Machine Unlearning is Extinction, Not Forgetting

In extinction, the original memory stays in place and a new inhibitory memory is layered on top. The neural
network version of that hypothesis: **the features still encode the forget class, and unlearning mostly
changed the read-out (the last layer)**. This notebook tests that with four cheap analyses:

| Analysis | How | What extinction predicts |
|---|---|---|
| **Linear probe** | Freeze the model, fit a *new* 100-way linear head on its features (including forget-class images), measure forget-class test accuracy | Unlearned models are probed **better than the retrained model** (excess = hidden knowledge) |
| **Head swap** | Put the original model's last layer on the unlearned body, and the reverse | Original head + unlearned body brings the class back: the suppression is in the head |
| **Weight change** | Relative change of each block's weights vs. the original | Changes concentrated in late layers / `fc` |
| **CKA similarity** | Representation similarity of each block to the original and the retrained model, on forget and retain images | Unlearned features stay closer to the **original** than to the retrained model |

Multi-context models (notebook 03, stage E) are analysed too, to see whether mixing contexts changes where
the forgetting happens.

**Time:** about 30 s–1 min per model, ~270 models → **1.5–2.5 h on T4 x2**.

## Kaggle settings for this notebook

Set these in the right-hand **Settings** panel (in the editor: *View → Show sidebar* if it is hidden).

| Setting | Value | Why |
|---|---|---|
| **Accelerator** | **GPU T4 x2** | Two GPUs: `ext_utils` runs one independent worker per GPU, so two models train at the same time. Your weekly quota is counted in session hours, not per GPU, so T4 x2 roughly doubles what you get per hour. (A single P100 also works: the code detects one GPU and uses one worker.) |
| **Internet** | **On** | Needed to download CIFAR-100 (~170 MB) and, in notebook 04, CIFAR-100-C, unless you attach them as datasets. Internet requires a phone-verified Kaggle account. |
| **Persistence** | Files only (optional) | Keeps `/kaggle/working` between *interactive* sessions, so you don't lose files if the browser disconnects. Not needed for background runs. |
| **Environment** | *Always use latest* for the first run, then **Pin to original environment** | Pinning freezes the library versions (PyTorch etc.), so a Kaggle image update can't break a half-finished experiment. |
| **Language** | Python | |

**Inputs to attach** (*Add Input* → *Your Work* / *Your Datasets*):

| Input | Why |
|---|---|
| `ext-code` dataset (contains `ext_utils.py`) | The shared library. Upload it once as a private dataset. |
| **CIFAR-100 Python** dataset ([fedesoriano/cifar100](https://www.kaggle.com/datasets/fedesoriano/cifar100)) | **Strongly recommended.** Without it CIFAR-100 is downloaded from cs.toronto.edu at ~50 kB/s (56 min in run #1). Check that the setup cell prints `CIFAR-100: found at …`. |
| Outputs of **notebooks 01, 02, 03** (`ext-01-originals`, `ext-02-retrain`, `ext-03-unlearning`) | The original, retrained and unlearned models. |
| *(when resuming)* this notebook's previous output (same name as below) | Skips finished jobs. |

**How to run.** Click **Save Version → Save & Run All (Commit)**. The notebook then runs in the background
for up to 12 hours with the browser closed. When it finishes, its files appear under *Output*, and the next
notebook attaches them as an input. 

**If a run stops early** (time budget or error), nothing is lost. Every finished job is written to disk
immediately. Attach this notebook's own latest output as an input and run it again: finished jobs are
skipped and their files are carried over.

## Setup: import the shared library

This cell finds `ext_utils.py` and imports it as `U`. All the real code (data pipeline, model, unlearning
methods, tests, job runner) lives there, so every notebook uses *identical* code.

`U.env_report()` then prints:
- the Python and PyTorch versions,
- which GPUs are visible (you should see **two Tesla T4** with the T4 x2 accelerator),
- free disk space,
- whether the internet is reachable,
- which inputs are attached.

**Check this output before going further.** If it says `GPU: none found`, the accelerator is off. If it
says `internet: OFF` and no CIFAR-100 dataset is attached, data loading will fail.

In [ ]:
# Locate ext_utils.py (from an attached dataset, a utility script, or the working folder) and import it.
import os, sys, glob
candidates = (glob.glob("/kaggle/working/ext_utils.py")
              + glob.glob("/kaggle/usr/lib/**/ext_utils.py", recursive=True)
              + glob.glob("/kaggle/input/**/ext_utils.py", recursive=True)
              + glob.glob(os.path.join(os.getcwd(), "ext_utils.py"))
              + glob.glob(os.path.join(os.getcwd(), "..", "ext_utils.py")))
if not candidates:
    raise FileNotFoundError(
        "ext_utils.py not found. Upload it as a private Kaggle Dataset (e.g. 'ext-code') and attach it "
        "with 'Add Input' -> 'Your Datasets' (see the Kaggle settings section above).")
sys.path.insert(0, os.path.dirname(os.path.abspath(candidates[0])))
import ext_utils as U
print("ext_utils loaded from:", U.UTILS_FILE)
U.env_report()

## Configuration

| Parameter | Value | What it does and why this value |
|---|---|---|
| `smoke` | `False` | `True` = a tiny dry run (30 training and 10 test images per class, 1 epoch, width 16) that finishes in minutes. Its files get a `_smoke` suffix, so they never mix with real results. Use it once to test the whole chain, then set `False`. **Must be the same in every notebook.** |
| `seeds` | `[0, 1, 2]` | Independent repetitions (different weight initialisation and data order). Three is the minimum for estimating variance. One statistical unit = (method, class, seed). |
| `forget_classes` | 10 classes | One class from each of 10 *different* CIFAR-100 superclasses, so no single superclass drives the result. The other 4 classes in each superclass are its "siblings" (semantic neighbours). |
| `norm` | `"bn"` | Normalization layer. `"bn"` = BatchNorm, the standard ResNet-18 of the unlearning literature (main experiments). `"gn"` = GroupNorm, a **control arm without running statistics**. The "BatchNorm illusion" (Kalani et al. 2026) showed that recomputing BatchNorm statistics alone can undo apparent forgetting. If the extinction signs also appear with GroupNorm, they are not a BatchNorm artifact. GroupNorm files get a `_gn` suffix, so both arms can live side by side. **Must be the same in every notebook of one arm.** |
| `pilot_class` | `"lamp"` | Used only for hyper-parameter calibration and the go/no-go pilot, then **excluded** from the main analysis. That way hyper-parameters are never tuned on the data that is later tested. |
| `width`, `epochs`, … | as in notebook 01 | Must match, so checkpoints load correctly. |
| `methods` | 5 methods | Which unlearned models to analyse. |
| `probe_epochs` | 30 | Passes of the linear-probe training over the 50,000 frozen feature vectors. A linear head converges quickly; 30 is plenty and takes seconds. |
| `probe_lr` | 1e-3 | Adam learning rate for the probe (features are standardised first, so a standard Adam rate works). |
| `cka_n_retain` | 500 | Number of retain-class test images used for CKA. CKA compares n×n similarity matrices, so a few hundred images give a stable estimate while keeping memory low. All 100 forget-class test images are always used. |
| `time_budget_h` | 11.0 | Kaggle stops sessions at 12 h. The runner stops *starting* new jobs after ~11 h (it estimates each job's length), so the notebook finishes cleanly and its output is saved. Re-run to continue. |
| `n_workers` | `None` | `None` = one worker process per visible GPU (2 on T4 x2, 1 on P100). Set 1 to force a single worker, e.g. to see tracebacks directly. |
| `reuse_ckpt` | `True` | If a job's checkpoint already exists (e.g. saved just before a crash), load it instead of recomputing it. |
| `raise_on_error` | `True` | After all jobs have run, the cell raises an error if any job failed, so failures can't go unnoticed. Finished jobs are always kept. |
| `poll_seconds` | 60 | How often the progress line (`done/total`, elapsed minutes, last log line of each worker) is printed. |

In [ ]:
CFG = U.make_cfg(
    smoke=False,
    seeds=[0, 1, 2],
    forget_classes=["maple_tree", "tiger", "rose", "shark", "bus",
                    "apple", "chair", "butterfly", "girl", "castle"],
    pilot_class="lamp",
    norm="bn",                      # "gn" = GroupNorm control arm (see notebook 01)
    width=64, epochs=40, batch_size=256, lr=0.1, pct_start=0.25,
    momentum=0.9, nesterov=True, weight_decay=5e-4, label_smoothing=0.0,
    amp=True, channels_last=True, cudnn_benchmark=True, eval_batch_size=1024,
    ckpt_fp16=True, retain_eval_subset=5000,
    methods=["FT", "NegGrad+", "RL", "SCRUB", "SalUn"],
    probe_epochs=30, probe_lr=1e-3, cka_n_retain=500,
    time_budget_h=11.0, n_workers=None, reuse_ckpt=True, raise_on_error=True, poll_seconds=60,
)
U.check_experiment_config(CFG)

## Run the mechanism analyses

Jobs cover every unlearned model plus the retrained and original model of each (class, seed). The original
model only gets the probe, which gives the upper bound of how probe-able the class is.

In [ ]:
jobs = U.jobs_mechanism(CFG)
print(f"{len(jobs)} models")
U.launch("mechanism", jobs, CFG)

## Preview

- **Probe:** `probe_forget_test_acc` for original ≥ unlearned > retrained would mean the knowledge is still
  in the features.
- **Head swap:** a high forget accuracy for `body_model+head_original` means the unlearned body still
  carries the class.

In [ ]:
df = U.load_df("mechanism", CFG)
if len(df):
    df = df[df["cls"] != CFG["pilot_class"]]
    display(df[df["test"] == "probe"].groupby(["kind", "method"])[["probe_forget_test_acc", "probe_test_acc"]].mean().round(4))
    hs = df[df["test"] == "headswap"]
    if len(hs):
        display(hs.pivot_table(index=["kind", "method"], columns="variant", values="forget_test_acc").round(4))
else:
    print("No results yet.")

## Next step

Save a version. Then open **notebook 06** and attach the outputs of notebooks **01–05**.

## Name of this run's output

Title this notebook **`ext-05-mechanism`**. When the run has finished, create a dataset from its output with the
same name: open the notebook's viewer page → **Output** → **New Dataset** → title **`ext-05-mechanism`**, Private.
(Or skip the dataset and attach the notebook output directly: *Add Input → Your Work → Notebooks*.)

| Output name | Contains | Attached by |
|---|---|---|
| `ext-05-mechanism` | `mechanism` results (probe, head swap, weight change, CKA) | notebook 06 |

- **Re-runs and resumes:** don't create a new name. Add a **New Version** of the same dataset. Each output
  already contains everything from earlier runs (it's carried over), so only the latest version matters.
  In the notebooks that use it, update the input to the new version.
- **Smoke (dry) runs:** add `-smoke`, e.g. `ext-05-mechanism-smoke`, so real and test outputs never share a dataset.
- **GroupNorm control arm** (`norm="gn"`): add `-gn`, e.g. `ext-05-mechanism-gn`.